In [132]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [133]:
artists = pd.read_json('artists.jsonl', lines=True)
genres_to_filter = ['pop', 'trap', 'rap', 'rock', 'metal', 'hip hop']

In [134]:
def filter_genres(genre_list, regex_pattern):
    return [genre for genre in genre_list if pd.Series([genre]).str.contains(regex_pattern, regex=True).any()]


In [135]:
filtered_artists = pd.DataFrame()
filtered_genres_dict = {}
for genre in genres_to_filter:
    regex_pattern = fr'\b({genre})\b'

    artists['filtered_genres'] = artists['genres'].apply(lambda x: filter_genres(x, regex_pattern))
    one_genre = artists[artists['filtered_genres'].apply(lambda x: x != [])]
    one_genre.loc[:, 'genres'] = genre
    filtered_artists = pd.concat([filtered_artists, one_genre], ignore_index=True)

# Drop the temporary 'filtered_genres' column
filtered_artists = filtered_artists.drop(columns=['filtered_genres'])

# Display the concatenated DataFrame
print(filtered_artists)

/tmp/ipykernel_714/77786420.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return [genre for genre in genre_list if pd.Series([genre]).str.contains(regex_pattern, regex=True).any()]
/tmp/ipykernel_714/77786420.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return [genre for genre in genre_list if pd.Series([genre]).str.contains(regex_pattern, regex=True).any()]


                          id                name   genres
0     7jVv8c5Fj3E9VhNjxT4snq           Lil Nas X      pop
1     0xRXCcSX89eobfrshSVdyu              MEDUZA      pop
2     4f7KfxeHq9BiylGmyXepGt      Tanishk Bagchi      pop
3     56mfhUDKa1vec6rSLZV5Eg           Jawsh 685      pop
4     6m6e7D2TnV0aYMllFFwMxu  Guilherme & Benuto      pop
...                      ...                 ...      ...
2277  50co4Is1HCEo8bhOyUWKpn          Young Thug  hip hop
2278  6U3ybJ9UHNKEdsH7ktGBZ7                 JID  hip hop
2279  1qQLhymHXFPtP5U8KNKsm6          Apache 207  hip hop
2280  6qgnBH6iDM91ipVXv28OMu          KAYTRANADA  hip hop
2281  46MWeeHNVMYRIIofQBEX98                BEAM  hip hop

[2282 rows x 3 columns]


In [136]:
len([ x for x in filtered_artists['genres'] if x == 'trap'])

268

In [137]:
filtered_artists.describe()

,id,name,genres
count,2282,2282,2282
unique,1391,1391,6
top,5gCRApTajqwbnHHPbr2Fpi,Juicy J,pop
freq,4,4,919


In [138]:
tracks = pd.read_json('tracks.jsonl', lines=True)
tracks.sample()

,id,name,popularity,duration_ms,explicit,id_artist,release_date,danceability,energy,key,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
15620,5j7gLuOsBOyqooruGxX4nx,Taking Over Me,55,228613,0,5nGIFgo0shDenQYSE0Sn7c,2003-03-04,0.476,0.796,9,-5.544,0.0335,0.0147,0.000029,0.0981,0.399,90.038


In [139]:
divided_songs = {}
for genre in genres_to_filter:
    genre_frame = filtered_artists[filtered_artists['genres'] == genre]
    genre_artist_ids = genre_frame['id'].tolist()
    divided_songs[genre] = tracks[tracks['id_artist'].apply(lambda x: x in genre_artist_ids)]

In [140]:
divided_songs['pop']

,id,name,popularity,duration_ms,explicit,id_artist,release_date,danceability,energy,key,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
29,1XPj5quoeV5Gd0paSUDpvm,You Send Me - Mono,61,165733,0,6hnWRPzGGKiapVX1UCdEAC,1958-02-01,0.572,0.365,4,-7.583,0.0276,0.8800,0.00000,0.1250,0.4100,96.022
45,2G0GextMwZJLkNxcSZ7ZJ3,(What A) Wonderful World - Mono,67,128787,0,6hnWRPzGGKiapVX1UCdEAC,1960-02,0.686,0.672,11,-5.523,0.0323,0.7000,0.00000,0.1350,0.8570,128.550
46,4yjz1aazw6R8ZURpGbCAkp,Bring It On Home To Me,65,165280,0,6hnWRPzGGKiapVX1UCdEAC,1960-01-01,0.482,0.569,0,-5.897,0.0328,0.6170,0.00000,0.1550,0.6090,70.843
48,0BFEyqJ9DJXS7gKg0Kj46R,You Send Me,64,165560,0,6hnWRPzGGKiapVX1UCdEAC,1960-01-01,0.553,0.291,4,-10.426,0.0301,0.8780,0.00000,0.1290,0.4070,96.217
50,5Fih0qlqrBjPIE0dFamuBr,(What A) Wonderful World,62,129293,0,6hnWRPzGGKiapVX1UCdEAC,1960-01-01,0.763,0.571,11,-10.552,0.0337,0.5290,0.00000,0.1540,0.9360,128.211
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22407,0LcNMuOiULmxJK3bdHTfDF,Missing You,65,214987,0,7gAppWoH7pcYmphCVTXkzs,2019-04-19,0.537,0.647,2,-7.370,0.0519,0.1680,0.00000,0.1080,0.5310,79.770
22408,1uviKYHZuM4uINK33F7sCt,Fix It to Break It,70,198799,0,7okSU80WTrn4LXlyXYbX3P,2020-03-27,0.493,0.461,2,-8.524,0.0456,0.8450,0.00000,0.1150,0.3500,51.414
22409,1fXmDeiCb3ABt5CzkMxp4u,Lotus Inn,63,195868,0,2jnIB6XdLvnJUeNTy5A0J2,2020-12-04,0.578,0.740,2,-5.080,0.0721,0.0119,0.00000,0.1800,0.4950,155.984
22410,27kcZEJvhkb1rzZS9gCpdA,remember the mornings,67,202355,0,7okSU80WTrn4LXlyXYbX3P,2020-11-27,0.590,0.443,0,-5.662,0.0722,0.6180,0.00000,0.1100,0.3420,149.447


## Wyliczanie współczynnika dla każdej piosenki względem sesji

Ile razy dana piosenka znalazła się w jednej sesji z innymi piosenkami zakwalifikowanymi do tej samej grupy gatunkowej.
Nie bierzemy pod uwagę piosenek przewiniętych "skip".

In [141]:
# sessions = pd.read_json('sessoins_part1.jsonl', lines=True)
# tracks.sample()

In [142]:
import pandas as pd
import json

filename = 'sessions_part1.jsonl'

# Read and clean the JSONL file
cleaned_lines = []
with open(filename, 'r', encoding='utf-8') as file:
    for line_number, line in enumerate(file, start=1):
        line = line.strip()
        if line:
            try:
                json_obj = json.loads(line)
                cleaned_lines.append(json_obj)
            except json.JSONDecodeError as e:
                print(f"Error parsing JSON at line {line_number}: {e}")
                break

# Create a DataFrame from the cleaned lines
sessions = pd.DataFrame(cleaned_lines)


In [143]:
sessions = sessions[sessions['event_type'] == 'play']


In [144]:
for genre in genres_to_filter:
    divided_songs[genre].loc[:, 'coherence'] = 0


/tmp/ipykernel_714/422349128.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  divided_songs[genre].loc[:, 'coherence'] = 0
/tmp/ipykernel_714/422349128.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  divided_songs[genre].loc[:, 'coherence'] = 0
/tmp/ipykernel_714/422349128.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/

In [154]:
def collect_track_ids(track_id, tracks_in_session, tracks_in_session_collected: set, last_track: list):
    if track_id in tracks_in_session and last_track[0] in tracks_in_session:
        tracks_in_session_collected.add(track_id)
        tracks_in_session_collected.add(last_track[0])
    last_track.pop()
    last_track.append(track_id)

In [151]:
divided_songs['rock'].loc[:, 'name']

11                    Blue Suede Shoes
16                Blue Moon - Take 9/M
17                        Tutti Frutti
20                      Blue Christmas
26                      Jailhouse Rock
                     ...              
22349                          Ophelia
22352                  Moth Into Flame
22392    Immortals - From "Big Hero 6”
22396                  Just Imagine It
22398                     Start A Riot
Name: name, Length: 7239, dtype: object

In [171]:
sessions_ids = set(sessions['session_id'].tolist())
for session_id in sessions_ids:
    tracks_in_session = sessions[sessions['session_id'] == session_id]['track_id'].tolist()
    for genre, track_list in divided_songs.items():
        tracks_in_session_collected = set()
        last_track = []
        last_track.append(track_list.iloc[0]['id'])
        track_list['id'].apply(lambda x: collect_track_ids(x, tracks_in_session, tracks_in_session_collected, last_track))
        if len(tracks_in_session) > 1:
            for track_id in tracks_in_session_collected:
                track_list.loc[track_list['id'] == track_id, 'coherence'] += 1

In [173]:
divided_songs['pop']['coherence'].apply(lambda x: print(x) if x > 4 else print('', end=''))

29       None
45       None
46       None
48       None
50       None
         ... 
22407    None
22408    None
22409    None
22410    None
22411    None
Name: coherence, Length: 11637, dtype: object

In [175]:
numeric_features = divided_songs['pop'].select_dtypes(include=['int64', 'float64']).columns
covariance_matrix = divided_songs['pop'][numeric_features].corr()
covariance_matrix.style.background_gradient(cmap='coolwarm', axis=None, vmin=-1, vmax=1).format(precision=3)

,popularity,duration_ms,explicit,danceability,energy,key,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,coherence
popularity,1.000,-0.130,0.070,0.064,-0.060,-0.011,0.032,0.020,-0.022,-0.015,-0.060,-0.031,-0.017,0.043
duration_ms,-0.130,1.000,-0.077,-0.166,-0.066,0.007,-0.121,-0.125,0.016,0.082,0.014,-0.171,-0.025,0.007
explicit,0.070,-0.077,1.000,0.188,0.004,0.000,0.002,0.376,-0.042,-0.030,-0.027,-0.043,0.002,-0.001
danceability,0.064,-0.166,0.188,1.000,0.138,0.010,0.134,0.166,-0.170,-0.065,-0.070,0.448,-0.178,-0.019
energy,-0.060,-0.066,0.004,0.138,1.000,0.019,0.737,0.119,-0.671,-0.080,0.158,0.448,0.159,-0.013
key,-0.011,0.007,0.000,0.010,0.019,1.000,0.011,0.027,-0.011,-0.002,0.002,0.033,0.009,0.011
loudness,0.032,-0.121,0.002,0.134,0.737,0.011,1.000,0.055,-0.522,-0.239,0.091,0.281,0.107,-0.016
speechiness,0.020,-0.125,0.376,0.166,0.119,0.027,0.055,1.000,-0.075,-0.049,0.061,0.071,0.090,-0.010
acousticness,-0.022,0.016,-0.042,-0.170,-0.671,-0.011,-0.522,-0.075,1.000,0.056,-0.045,-0.272,-0.115,0.020
instrumentalness,-0.015,0.082,-0.030,-0.065,-0.080,-0.002,-0.239,-0.049,0.056,1.000,-0.038,-0.105,0.000,0.020


In [184]:
from sklearn.feature_selection import mutual_info_regression
import numpy as np
# mutual_info_regression(divided_songs['rock'].loc[:, ('popularity', 'duration_ms', 'danceability',
#                                                      'energy', 'loudness', 'speechiness', 'acousticness', 'liveness',
#                                                      'valence', 'tempo')], divided_songs['rock']['coherence'])
columns = ['popularity', 'duration_ms', 'danceability', 'energy', 'loudness',
          'speechiness', 'acousticness', 'liveness', 'valence', 'tempo', 'genre']
mutual_info = pd.DataFrame(columns=columns)
for genre in genres_to_filter:
    new_data = mutual_info_regression(divided_songs[genre].loc[:, columns[:-1]], divided_songs[genre]['coherence'])
    new_data = np.append(new_data, genre)
    new_data = new_data.reshape(1, len(new_data))
    new_rows_df = pd.DataFrame(new_data, columns=mutual_info.columns, )
    mutual_info = pd.concat([mutual_info, new_rows_df], ignore_index=True)
mutual_info

,popularity,duration_ms,danceability,energy,loudness,speechiness,acousticness,liveness,valence,tempo,genre
0,0.0033009468294604716,0.0,0.003331525973843341,0.0,0.0019182418969254655,0.00018992361117753376,0.003406220307297758,0.0,0.0,0.0022328154074262585,pop
1,0.0,0.0,0.0,0.0020647734965386633,0.0,0.0010260529876884306,0.0035986651114745527,0.015673843728713166,0.0003617239923308091,0.002971820722757812,trap
2,0.010439080386653643,0.002800751595465556,0.0,0.0011101227574279449,0.0008132384769723444,0.0,0.0,0.00584350943890577,0.002349719935974548,0.0006450966494249144,rap
3,0.0,0.0,0.0027129230070777055,0.0,0.0007059593390739138,0.0003900415817650327,0.0010647639975331913,0.00954312733581908,2.195537614824161e-06,0.00028611639219455753,rock
4,0.0,0.003489762271179586,0.0,0.0,0.0027312916089865524,0.010066238294517582,0.001955194666436455,0.006469927480249105,0.0027253615675979503,0.004767361518865698,metal
5,0.0038877962733403493,0.0012756186561828642,6.25655404391523e-05,0.0022703531467413995,0.0,0.0,0.006409930776154482,0.010744172953674358,0.0,0.0,hip hop
